# 50 -- enumeration probe, step 3: fit probes via CV, pick the best layer per task

No GPU -- consumes `02`'s dumped hidden states (`feats_fit_full.npy` / `meta_fit_full.csv` /
`layers_fit_full.json`) plus `01`'s fold assignments. Two independent tasks, each swept and
selected on its OWN metric:

- **counting** (`number`): for every kept layer, video-grouped 5-fold CV over the three
  agreed classifier kinds in order -- `multinomial` (rung 34's original recipe), `balanced`
  (`class_weight="balanced"`), `ordinal` (Frank & Hall) -- and a small `C` grid. Scored by
  out-of-fold exact-match accuracy on the raw gold count.
- **fo_class**: for every kept layer and `C`, the same 5-fold CV with the one-vs-rest
  multi-label probe. Scored by out-of-fold EXACT SET MATCH -- the real challenge metric,
  not per-label accuracy (see `multilabel_probe.py`'s docstring on the multiplicative
  collapse this produces).

The winning (layer, kind, C) per task is what `04_final_read.ipynb` refits on the FULL fit
pool and reads once against the held-out final-read set.

In [ ]:
# --- parameters (RAW LITERALS ONLY -- papermill injects a new cell right after THIS one) --
HIDDEN_DIR = "/workspace/repo/experiments/50-enumeration-probe/runs/50_hidden_v1"
TAG = "full"
C_GRID = [0.01, 0.1, 1.0]
PCA_N_COMPONENTS = 128
SEED = 0
LATE_LAYER_MIN = 18  # rung 34's own finding: counting becomes decodable around layer 18-24;
# layers below this are known to carry no signal (and layer 0 is degenerate -- the hidden
# state at the fixed last-prompt-token position is content-independent before any attention
# has mixed in the image/question, so its PCA has exactly zero total variance).

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "50-enumeration-probe":
    EXP = REPO / "experiments" / "50-enumeration-probe"

if str(EXP / "_tools") not in sys.path:
    sys.path.insert(0, str(EXP / "_tools"))

import ordinal_probe as OP
import multilabel_probe as MP

HIDDEN_DIR = Path(HIDDEN_DIR)
print("repo:", REPO, "| exp:", EXP, "| hidden_dir:", HIDDEN_DIR)

In [ ]:
# --- 1. load the fit-pool dump + manifests, align rows by qID ---------------------
feats = np.load(HIDDEN_DIR / f"feats_fit_{TAG}.npy")  # (n, n_layers, dim)
meta = pd.read_csv(HIDDEN_DIR / f"meta_fit_{TAG}.csv")  # qID, answer_format -- X's row order
layers = json.loads((HIDDEN_DIR / f"layers_fit_{TAG}.json").read_text())
assert feats.shape[0] == len(meta), "feats/meta row-count mismatch -- dump is corrupt"
assert feats.shape[1] == len(layers), "feats/layers layer-count mismatch -- dump is corrupt"

keep_idx = [i for i, l in enumerate(layers) if l >= LATE_LAYER_MIN]
assert keep_idx, f"no kept layer >= {LATE_LAYER_MIN} in {layers}"
feats = feats[:, keep_idx, :]
layers = [layers[i] for i in keep_idx]
print(f"restricted to {len(layers)} late layers (>= {LATE_LAYER_MIN}): {layers}")

num_manifest = pd.read_csv(EXP / "RESULTS_fit_number_v1.csv")
fo_manifest = pd.read_csv(EXP / "RESULTS_fit_foclass_v1.csv")

meta_pos = {q: i for i, q in enumerate(meta["qID"])}


def _positions(manifest: pd.DataFrame) -> np.ndarray:
    missing = [q for q in manifest["qID"] if q not in meta_pos]
    assert not missing, f"{len(missing)} manifest qIDs missing from the dump (e.g. {missing[:5]})"
    return np.array([meta_pos[q] for q in manifest["qID"]])


num_pos = _positions(num_manifest)
fo_pos = _positions(fo_manifest)
print(f"counting: {len(num_pos)} rows located in the dump")
print(f"fo_class: {len(fo_pos)} rows located in the dump")

In [ ]:
# --- 2. counting task: CV sweep over (layer, kind, C) ------------------------------
y_num = num_manifest["answer"].astype(int).to_numpy()
fold_num = num_manifest["fold"].to_numpy()
X_num_all = feats[num_pos]  # (n_number, n_layers, dim)

num_rows = []
for li, layer in enumerate(layers):
    X_layer = X_num_all[:, li, :].astype("float32")
    for kind in ("multinomial", "balanced", "ordinal"):
        for C in C_GRID:
            oof_pred = np.empty(len(y_num), dtype=int)
            for f in np.unique(fold_num):
                tr, te = fold_num != f, fold_num == f
                fr, _, proba_te = OP.fit(X_layer[tr], y_num[tr], X_layer[te], kind=kind,
                                         layer=layer, C=C, n_components=PCA_N_COMPONENTS, seed=SEED)
                oof_pred[te] = OP.predict(proba_te, fr.classes)
            acc = float((oof_pred == y_num).mean())
            num_rows.append({"layer": layer, "kind": kind, "C": C, "cv_exact_acc": acc})
    print(f"layer {layer} done ({li + 1}/{len(layers)})", flush=True)

num_cv = pd.DataFrame(num_rows).sort_values("cv_exact_acc", ascending=False).reset_index(drop=True)
print(num_cv.head(10))

In [ ]:
# --- 3. fo_class task: CV sweep over (layer, C), scored by EXACT SET MATCH ---------
fo_answers = fo_manifest["answer"].to_numpy()
fold_fo = fo_manifest["fold"].to_numpy()
X_fo_all = feats[fo_pos]  # (n_foclass, n_layers, dim)
vocab = MP.build_vocab(fo_answers)  # from the fit pool only -- 04 re-checks read-time coverage
print("vocab:", vocab)

fo_rows = []
for li, layer in enumerate(layers):
    X_layer = X_fo_all[:, li, :].astype("float32")
    for C in C_GRID:
        oof_exact = np.empty(len(fo_answers), dtype=bool)
        for f in np.unique(fold_fo):
            tr, te = fold_fo != f, fold_fo == f
            _, _, proba_te = MP.fit(X_layer[tr], fo_answers[tr], X_layer[te], vocab=vocab,
                                    layer=layer, C=C, n_components=PCA_N_COMPONENTS, seed=SEED)
            pred_sets = MP.predict_sets(proba_te, vocab)
            oof_exact[te] = MP.exact_set_match(pred_sets, fo_answers[te])
        fo_rows.append({"layer": layer, "C": C, "cv_exact_set_acc": float(oof_exact.mean())})
    print(f"layer {layer} done ({li + 1}/{len(layers)})", flush=True)

fo_cv = pd.DataFrame(fo_rows).sort_values("cv_exact_set_acc", ascending=False).reset_index(drop=True)
print(fo_cv.head(10))

In [ ]:
# --- 4. persist: full CV grids + the winning config per task -----------------------
num_cv.to_csv(EXP / "RESULTS_probe_number_cv.csv", index=False)
fo_cv.to_csv(EXP / "RESULTS_probe_foclass_cv.csv", index=False)

best_num = num_cv.iloc[0].to_dict()
best_fo = fo_cv.iloc[0].to_dict()
best_config = {
    "counting": {"layer": int(best_num["layer"]), "kind": best_num["kind"],
                 "C": float(best_num["C"]), "cv_exact_acc": best_num["cv_exact_acc"]},
    "fo_class": {"layer": int(best_fo["layer"]), "C": float(best_fo["C"]),
                 "cv_exact_set_acc": best_fo["cv_exact_set_acc"], "vocab": vocab},
    "pca_n_components": PCA_N_COMPONENTS,
    "seed": SEED,
}
(EXP / "RESULTS_best_config.json").write_text(json.dumps(best_config, indent=2))
print(json.dumps(best_config, indent=2))